# ED Alignment Algorithm on Real Piano Performances

This notebook evaluates the ED alignment algorithm in `compareMusic` on real piano performances from the [(n)ASAP: the (note-)Aligned Scores And Performances dataset](https://github.com/CPJKU/asap-dataset) (Peter et al., 2023).

(n)ASAP is a dataset of aligned musical scores and performances built by extending the ASAP dataset with note-level annotations. The ASAP contains 236 distinct musical scores and 1067 performances of Western classical piano music from 15 different composers, the piece directory contains the XML and MIDI score, plus all of the performances of a specific piece, including: 
- `midi_score`: the MIDI score (what should be played) — used as **reference**
- `midi_performance`: what the pianist actually played — used as **response**
- `note_alignments.tsv`: official note-level alignment annotations — used as **ground truth**

bib citation:

@article{Peter-2023,
 title = {Automatic Note-Level Score-to-Performance Alignments in the ASAP Dataset},
 author = {Peter, Silvan David and Cancino-Chacón, Carlos Eduardo and Foscarin, Francesco and McLeod, Andrew Philip and Henkel, Florian and Karystinaios, Emmanouil and Widmer, Gerhard},
 doi = {10.5334/tismir.149},
 journal = {Transactions of the International Society for Music Information Retrieval {(TISMIR)}},
 year = {2023}
}

relevant paper: https://transactions.ismir.net/articles/10.5334/tismir.149#5-alignment-of-the-asap-dataset

## Download the ASAP Dataset

In [1]:
import os

# Note: use CPJKU version (not fosfrancesco) because only CPJKU has
# the note_alignments TSV files are needed for ground truth comparison.
if not os.path.exists("asap-dataset"):
    os.system("git clone https://github.com/CPJKU/asap-dataset.git")
else:
    print("asap-dataset already exists, skipping download.")

ASAP_PATH = "asap-dataset"

asap-dataset already exists, skipping download.


## Load the ground truth

The `note_alignment.tsv` file in each performance contains the official note-level alignment annotations. Each row in the TSV represents one note pair. 

The `label` has three possible values:
- `match`: a score note (reference) and a performance note (response) were successfully aligned
- `deletion`: the score note was not played in the performance
- `insertion`: an extra note was played that has no counterpart in the score

For `match` and `insertion` rows, the TSV provides `onset` (onset time in seconds) and `pitch` (MIDI pitch) for the performance note.

In [2]:
import csv

def load_ground_truth(tsv_path):
    """
    Read a note_alignments TSV file from the CPJKU ASAP dataset.

    The TSV columns are: xml_id, midi_id, track, channel, pitch, onset
        - if xml_id == 'insertion': extra note played with no score counterpart
        - if midi_id == 'deletion': score note not played in the performance
        - otherwise: matched pair, onset and pitch refer to the performance note
    
    Args:
        tsv_path: str, path to the note_alignments.tsv file

    Returns:
        list of dicts, each with keys:
            label  -> 'paired', 'insertion', or 'deletion'
            onset  -> float (seconds) or None for deletions
            pitch  -> int (MIDI pitch number) or None for deletions
    """
    rows = []
    with open(tsv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            xml_id  = row["xml_id"].strip()
            midi_id = row["midi_id"].strip()

            if midi_id == "deletion":
                rows.append({"label": "deletion", "onset": None, "pitch": None})
            elif xml_id == "insertion":
                rows.append({
                    "label": "insertion",
                    "onset": float(row["onset"]),
                    "pitch": int(row["pitch"]),
                })
            else:
                rows.append({
                    "label": "paired",
                    "onset": float(row["onset"]),
                    "pitch": int(row["pitch"]),
                })
    return rows

## Load Score (reference)/Performance (response) pairs from ASAP

Helper functions for format conversion: These functions convert a MIDI file into the `{pitch, start, duration}` format used by `compare_MIDI.py`.

In [3]:
import pretty_midi

def midi_file_to_notes(midi_path):
    """
    Parse a MIDI file and return all its notes in compareMusic format.
    
    Args:
        midi_path: str, path to the MIDI file

    Returns:
        list of dicts sorted by onset time, each with keys:
            pitch -> int, MIDI note number
            start -> float, onset time in seconds
            duration -> float, note length in seconds
    """
    midi_data = pretty_midi.PrettyMIDI(midi_path)
    all_notes = []
    for instrument in midi_data.instruments:
        if instrument.is_drum: # Ignore drum tracks
            continue
        for note in instrument.notes:
            all_notes.append({
                "pitch": note.pitch,
                "start": round(note.start, 3),
                "duration": round(note.end - note.start, 3),
            })
    all_notes.sort(key=lambda n: (n["start"], n["pitch"]))
    return all_notes


def build_sample(ref_path, response_path, composer, title, metadata_row):
    """
    Convert one score/performance MIDI pair into a sample dict
    ready for compare_performance_ED.

    Args:
        ref_path: str, path to the score (reference) MIDI file
        response_path: str, path to the performance (response) MIDI file
        composer: str
        title: str
        metadata_row: dict, one row from metadata.csv

    Returns:
        sample dict with keys: composer, title, reference, response, metadata_row
        Returns None if either MIDI file produces zero notes.
    """
    score_notes = midi_file_to_notes(ref_path)
    perf_notes  = midi_file_to_notes(response_path)

    if not score_notes or not perf_notes:
        return None

    return {
        "composer": composer,
        "title": title,
        "reference": {"notes": score_notes},
        "response": {"notes": perf_notes},
        "metadata_row": metadata_row,
    }

For the ASAP dataset, the `metadata.csv` lists every score/performance pair in the dataset, check that both MIDI files exist on disk.

In [4]:
def load_samples(asap_path, composer):
    """
    Read the ASAP metadata CSV and return a list of sample dicts
    for a specific composer only.

    Args:
        asap_path: str, path to the cloned ASAP repo
        composer: str, e.g. "Bach"

    Returns:
        list of sample dicts (see build_sample)
    """
    metadata_path = os.path.join(asap_path, "metadata.csv")
    samples = []

    with open(metadata_path, "r", encoding="utf-8") as csv_file:
        reader = csv.DictReader(csv_file)
        for row in reader:
            if row.get("composer", "").strip() == composer:
                ref_path = os.path.join(asap_path, row.get("midi_score", "").strip())
                response_path = os.path.join(asap_path, row.get("midi_performance", "").strip())

                if os.path.isfile(ref_path) and os.path.isfile(response_path):
                    sample = build_sample(
                        ref_path, response_path,
                        row.get("composer", "Unknown"),
                        row.get("title", "Unknown"),
                        dict(row),
                    )
                    if sample is not None:
                        samples.append(sample)

    return samples

samples = load_samples(ASAP_PATH, "Bach")

print("Loaded", len(samples), "score (reference) /performance (response) pairs.")
for i, sample in enumerate(samples[:10]):
    print(f"{str(i + 1)}. {sample['composer']}({sample['title']}) |"
          f" ref notes: {len(sample['reference']['notes'])} |"
          f" response notes: {len(sample['response']['notes'])}")

/Users/jz7125/compareMusic/.venv/lib/python3.13/site-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(


Loaded 169 score (reference) /performance (response) pairs.
1. Bach(Fugue_bwv_846) | ref notes: 755 | response notes: 754
2. Bach(Fugue_bwv_848) | ref notes: 1429 | response notes: 1435
3. Bach(Fugue_bwv_848) | ref notes: 1429 | response notes: 1438
4. Bach(Fugue_bwv_848) | ref notes: 1429 | response notes: 1429
5. Bach(Fugue_bwv_848) | ref notes: 1429 | response notes: 1431
6. Bach(Fugue_bwv_848) | ref notes: 1429 | response notes: 1433
7. Bach(Fugue_bwv_848) | ref notes: 1429 | response notes: 1438
8. Bach(Fugue_bwv_848) | ref notes: 1429 | response notes: 1440
9. Bach(Fugue_bwv_848) | ref notes: 1429 | response notes: 1431
10. Bach(Fugue_bwv_848) | ref notes: 1429 | response notes: 1430


# Run Alignment on Each Pair

Passing each score/performance pair through `compare_performance_ED`, where score MIDI is the reference and the performance MIDI is the response.

**Why do we need the normalised events? - to fix the onset mismatch bug**
- Inside `compare_performance_ED`, `normalize_start_times` shifts the first note to t = 0, and `group_notes_into_events` groups simultaneous notes into chords. The `response_index` values stored in `event_details` refer to positions in these grouped events, not in the original flat note list. Therefore, need to reproduce these two steps here so that the correct onset and pitch for each event during ground truth comparison can be looked up.

In [5]:
from evaluation_function.compare_MIDI import (
    compare_performance_ED,
    normalize_start_times,
    group_notes_into_events,
)


def run_alignment_on_samples(samples):
    """
    Run compare_performance_ED on every sample and collect results.

    We also store:
      - response_events_normalized: the grouped event list after start-time
        normalisation, needed to look up correct onset/pitch during evaluation.
      - response_onset_offset: the original first-note onset time, needed to
        convert normalised onsets back to absolute time for GT comparison.

    Args:
        samples: list of sample dicts from load_samples()

    Returns:
        list of result dicts, each with keys:
            composer, title, stats, event_details, is_correct,
            response_events_normalized, response_onset_offset
    """
    results = []
    for sample in samples:
        result = compare_performance_ED(
            sample["response"],
            sample["reference"],
        )

        # Reproduce the same normalisation + grouping done inside the pipeline.
        # This gives us the event list whose indices match event_details.
        response_notes_norm = normalize_start_times(sample["response"]["notes"])
        response_events_norm = group_notes_into_events(response_notes_norm)

        # Record the offset that normalize_start_times subtracted from every onset.
        # Adding it back can convert a normalised onset to the original onset.
        if sample["response"]["notes"]:
            response_onset_offset = sample["response"]["notes"][0]["start"]
        else:
            response_onset_offset = 0.0

        results.append({
            "composer": sample["composer"],
            "title": sample["title"],
            "stats": result.stats,
            "event_details": result.event_details,
            "is_correct": result.is_correct,
            "response_events_normalized": response_events_norm,
            "response_onset_offset": response_onset_offset,
        })

    return results

all_results = run_alignment_on_samples(samples)
print("Alignment complete for", len(all_results), "pieces.")

Alignment complete for 169 pieces.


## Compute Precision, Recall, and F1 

Compares our alignment output against the ASAP ground truth to compute standard information retrieval metrics:

- **Precision**: TP / (TP + FP) 
        - i.e. of all note pairs we called a match, how many the ground truth also call a match
- **Recall**: TP / (TP + FN)
        - i.e of all note pairs the ground truth called a match, how many we find
- **F1**: harmonic mean of precision and recall

A predicted match is a True Positive only if both pitch and onset time (rounded to 1 decimal place) agree with the ground truth.

**fix the index problem caused by the grouping in the pipeline**: look up pitches and onsets from `response_events_normalized` (the grouped events) instead of the flat note list. Also, need to add back `response_onset_offset` to every onset so that the normalised times match the original times stored in the ground truth TSV

In [6]:
def convert_pipeline_output(event_details, response_events_normalized):
    """
    Convert event_details from compare_performance_ED into a list of
    dicts with keys: onset, pitch, label.
    Labels follow the TSV convention:
        'paired': correctly aligned note pair (pitch matches)
        'deletion' : score note not found in response (missing)
        'insertion': response note has no score counterpart (extra)

    Chord events are expanded note-by-note using correct_pitches /
    missing_pitches / extra_pitches from event_level_feedback.

    Args:
        event_details: list of dicts from compare_performance_ED
        response_events_normalized: list of EVENT dicts produced by
                                    group_notes_into_events on the normalised
                                    response notes — NOT the flat note list.

    Returns:
        list of dicts with keys:
            onset -> float (normalised seconds) or None for deletions
            pitch -> int (MIDI pitch) or None for deletions
            label -> 'paired', 'insertion', or 'deletion'
    """
    my_pairs = []

    for event in event_details:
        op = event["operation_type"]

        if event["event_type"] == "note":
            if op in ("match", "replacement"):
                # Look up onset and pitch from the grouped event list.
                ev = response_events_normalized[event["response_index"] - 1]
                my_pairs.append({
                    "onset": ev["event_start"],
                    "pitch": ev["notes"][0]["pitch"],
                    "label": "paired",
                })
            elif op == "missing":
                my_pairs.append({"onset": None, "pitch": None, "label": "deletion"})
            elif op == "extra":
                ev = response_events_normalized[event["response_index"] - 1]
                my_pairs.append({
                    "onset": ev["event_start"],
                    "pitch": ev["notes"][0]["pitch"],
                    "label": "insertion",
                })

        elif event["event_type"] == "chord":
            if op == "missing":
                # Entire chord missing: one deletion per note in the ref chord
                # correct_pitches + missing_pitches = all ref notes in this chord
                num_ref_notes = (
                    len(event["correct_pitches"] or [])
                    + len(event["missing_pitches"] or [])
                )
                for i in range(num_ref_notes):
                    my_pairs.append({"onset": None, "pitch": None, "label": "deletion"})
            elif op == "extra":
                # Every note in the extra response chord counts as an insertion.
                ev = response_events_normalized[event["response_index"] - 1]
                for note in ev["notes"]:
                    my_pairs.append({
                        "onset": ev["event_start"],
                        "pitch": note["pitch"],
                        "label": "insertion",
                    })
            else:
                # Aligned chord pair (match or replacement).
                ev = response_events_normalized[event["response_index"] - 1]
                onset = ev["event_start"]

                # Build a pitch-class -> MIDI-pitch lookup from the response chord.
                # This fixes the bug where the original code always used the first
                # note's pitch regardless of which pitch class was being expanded.
                pc_to_midi = {}
                for note in ev["notes"]:
                    pc = note["pitch"] % 12
                    pc_to_midi[pc] = note["pitch"]

                # Correctly matched pitch classes -> paired
                for pc in (event["correct_pitches"] or []):
                    midi_pitch = pc_to_midi.get(pc, pc)  # fallback if lookup fails
                    my_pairs.append({"onset": onset, "pitch": midi_pitch, "label": "paired"})

                # Reference pitch classes not played -> deletion
                for pc in (event["missing_pitches"] or []):
                    my_pairs.append({"onset": None, "pitch": None, "label": "deletion"})

                # Response pitch classes with no reference counterpart -> insertion
                for pc in (event["extra_pitches"] or []):
                    midi_pitch = pc_to_midi.get(pc, pc)
                    my_pairs.append({"onset": onset, "pitch": midi_pitch, "label": "insertion"})

    return my_pairs

In [7]:
def compute_metrics(ground_truth, my_pairs, onset_offset=0.0):
    """
    Compare pipeline alignment against ASAP ground truth.

    A predicted match is a True Positive only if both pitch and
    onset time (rounded to 1 decimal place) agree with the GT.

    Args:
        ground_truth: list of dicts from load_ground_truth()
        my_pairs: list of dicts from convert_pipeline_output()
        onset_offset:  float, the value that was subtracted from all response
                       onsets by normalize_start_times. Added back here so that
                       our normalised onsets match the absolute GT onsets.

    Returns:
        dict with keys: precision, recall, f1, tp, fp, fn
    """
    # Build a set of GT matched (pitch, onset) pairs
    gt_matches = set()
    for row in ground_truth:
        if row["label"] == "paired":
            gt_matches.add((int(row["pitch"]), round(float(row["onset"]), 1)))

    # Build a set of our matched (pitch, onset) pairs.
    # Add the onset_offset back so our normalised times match GT absolute times.
    my_matches = set()
    for row in my_pairs:
        if row["label"] == "paired" and row["onset"] is not None:
            absolute_onset = row["onset"] + onset_offset
            my_matches.add((int(row["pitch"]), round(float(absolute_onset), 1)))

    tp = len(my_matches & gt_matches)
    fp = len(my_matches - gt_matches)
    fn = len(gt_matches - my_matches)

    if tp + fp > 0:
        precision = tp / (tp + fp)
    else:
        precision = 0.0

    if tp + fn > 0:
        recall = tp / (tp + fn)
    else:
        recall = 0.0

    if precision + recall > 0:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = 0.0

    return {
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "tp": tp, "fp": fp, "fn": fn,
    }

Evaluate against ground truth:

In [8]:
def evaluate_one_piece(asap_path, metadata_row, event_details,
                       response_events_normalized, response_onset_offset):
    """
    Run ground truth comparison for each score/performance pair.

    Args:
        asap_path: str, path to ASAP repo root
        metadata_row: dict, one row from metadata.csv
        event_details: list from compare_performance_ED result
        response_events_normalized: list of event dicts (normalized + grouped)
        response_onset_offset: float, onset of the first response note
                               before normalization — added back to convert
                               normalised onsets to absolute times for GT comparison.

    Returns:
        metrics dict, or None if the TSV file is not found
    """
    tsv_rel = metadata_row.get("note_alignments", "").strip()
    tsv_path = os.path.join(asap_path, tsv_rel)

    if not os.path.isfile(tsv_path):
        print("TSV not found:", tsv_path)
        return None

    ground_truth = load_ground_truth(tsv_path)
    my_pairs = convert_pipeline_output(event_details, response_events_normalized)
    metrics = compute_metrics(ground_truth, my_pairs, onset_offset=response_onset_offset)

    return metrics

In [9]:
import pandas as pd

eval_rows = []
for sample, result in zip(samples, all_results):
    metrics = evaluate_one_piece(
        ASAP_PATH,
        sample["metadata_row"],
        result["event_details"],
        result["response_events_normalized"],
        result["response_onset_offset"],
    )
    if metrics is not None:
        eval_rows.append({
            "Piece": result["composer"] + " (" + result["title"] + ")",
            "Precision": metrics["precision"],
            "Recall": metrics["recall"],
            "F1": metrics["f1"],
            "TP": metrics["tp"],
            "FP": metrics["fp"],
            "FN": metrics["fn"],
        })

df_eval = pd.DataFrame(eval_rows)
display(df_eval.head(10))

print("Mean Precision:", round(df_eval["Precision"].mean(), 4))
print("Mean Recall :", round(df_eval["Recall"].mean(), 4))
print("Mean F1 :", round(df_eval["F1"].mean(), 4))

,Piece,Precision,Recall,F1,TP,FP,FN
0,Bach (Fugue_bwv_846),0.9444,0.8740,0.9078,645,38,93
1,Bach (Fugue_bwv_848),0.9614,0.9322,0.9465,1319,53,96
2,Bach (Fugue_bwv_848),0.9376,0.9031,0.9200,1277,85,137
3,Bach (Fugue_bwv_848),0.9494,0.9171,0.9329,1294,69,117
4,Bach (Fugue_bwv_848),0.9370,0.9083,0.9224,1278,86,129
5,Bach (Fugue_bwv_848),0.9470,0.9190,0.9328,1304,73,115
6,Bach (Fugue_bwv_848),0.9585,0.9294,0.9437,1316,57,100
7,Bach (Fugue_bwv_848),0.9453,0.9071,0.9258,1279,74,131
8,Bach (Fugue_bwv_848),0.9563,0.9273,0.9416,1313,60,103
9,Bach (Fugue_bwv_848),0.9541,0.9251,0.9394,1310,63,106


Mean Precision: 0.941
Mean Recall : 0.9044
Mean F1 : 0.9221
